# Concurrent Load Evaluator

Measures **throughput and latency under parallel request load**.

For each (context, concurrency level) pair the evaluator fires N simultaneous threads and records:
- Average per-request latency
- Throughput (requests / second)
- Average prompt and completion token counts
- HTTP status code distribution

The evaluator is **data-agnostic**: it accepts any `Dict[str, str]` mapping a label to a
context string. The `context_utils` module provides helpers to build that dict from any source.

## 1. Imports

In [ ]:
from bellmira.evaluators import ModelParallelLoadEvaluator
from bellmira.utils.context_utils import (
    contexts_from_word_counts,
    contexts_from_files,
    contexts_from_bible,
    read_text_file,
)

## 2. Build contexts

Pick the approach that fits your data source.

### Option A — word-count truncation (any text)

In [ ]:
MY_TEXT_PATH = "/workspaces/BeLLMira/resources/data/text/bible.txt"  # replace with any .txt
text = read_text_file(MY_TEXT_PATH)

contexts = contexts_from_word_counts(
    text,
    word_counts=[500, 1500, 4000],
)
print({k: f"{len(v.split())} words" for k, v in contexts.items()})

### Option B — one file per context

In [ ]:
# contexts = contexts_from_files([
#     "/path/to/doc_short.txt",
#     "/path/to/doc_medium.txt",
#     "/path/to/doc_long.txt",
# ])

### Option C — Bible corpus (bundled)

In [ ]:
# contexts = contexts_from_bible(
#     bible_path="/workspaces/BeLLMira/resources/data/text/bible.txt",
#     chapter_numbers=[2, 5, 14],
# )

## 3. Configuration

In [ ]:
MODEL_URL = "https://millmchatqwen25-backend.dat-aip-millm.qa.mbcp.cloud/"

# Prompts are cycled across threads; keep several for variety
PROMPTS = [
    "Summarise the key events described in this passage.",
    "What is the main theme of the text above?",
    "Who are the main characters and what do they do?",
    "What lessons can be drawn from this passage?",
]

# Concurrency levels to sweep; evaluation stops on first error
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32]

## 4. Initialise the evaluator

In [ ]:
evaluator = ModelParallelLoadEvaluator(
    url=MODEL_URL,
    prompts=PROMPTS,
    contexts=contexts,
    concurrency_levels=CONCURRENCY_LEVELS,
    temperature=0.0,
)

## 5. Run evaluation

In [ ]:
results = evaluator.evaluate()
results

## 6. Inspect throughput and latency per concurrency level

In [ ]:
for concurrency_key, token_data in results.items():
    print(f"\n=== {concurrency_key} ===")
    for token_label, stats in token_data.items():
        print(f"  [{token_label}] latency={stats['Avg_latency']}s  "
              f"throughput={stats['Throughput']} req/s  "
              f"completion_tokens={stats['Avg_completion_tokens']}")

## 7. Extract flat metrics for logging

In [ ]:
metrics = evaluator.extract_threshold_metrics(results)
for key, val in metrics.items():
    print(f"{key}: {val}")

## 8. Run via Experiments (multi-model)

In [ ]:
from bellmira.llm_experiments.experiments import Experiments

evaluator_args = {
    "prompts": PROMPTS,
    "contexts": contexts,
    "concurrency_levels": CONCURRENCY_LEVELS,
}

experiments_list = [
    {
        "model": "Qwen3-4B",
        "url": "https://chatqwen3-4-backend.dat-aip-millm.qa.mbcp.cloud/",
        "env": "QA",
        "vllm_version": "0.9.2",
        "evaluator": ModelParallelLoadEvaluator,
        "evaluator_args": evaluator_args,
        "output_filename": "concurrent_load_results.csv",
    },
    {
        "model": "Qwen3-14B",
        "url": "https://chatqwen3-14-backend.dat-aip-millm.qa.mbcp.cloud/",
        "env": "QA",
        "vllm_version": "0.9.2",
        "evaluator": ModelParallelLoadEvaluator,
        "evaluator_args": evaluator_args,
        "output_filename": "concurrent_load_results.csv",
    },
]

experiments = Experiments(experiments_list, vllm_version="0.9.2")
experiments.create_experiments()
experiments.launch_experiments()